# M3L3 E19 — Sistema empresarial completo con evaluador automático
### Módulo 3 · Lecture 3 · Sistemas Multiagente Avanzados

**Caso terminado:** sistema empresarial con 4 departamentos especializados, un evaluador automático de calidad y soporte opcional de trazabilidad con Langfuse.

## ¿Qué vas a ver en este ejercicio?
- 4 departamentos corporativos: HR, IT, Finance, Legal.
- Un **evaluador automático** que audita cada respuesta en 4 dimensiones (relevancia, completitud, precisión, claridad).
- Integración opcional con **Langfuse** para trazabilidad de producción.

## Arquitectura del sistema

> **Auto-evaluador:** nodo que recibe la respuesta de un agente especialista y la audita usando el mismo LLM. Produce métricas de calidad sin intervención humana.

```
START
  |
  v
enterprise_router_node
  |       |         |        |        |
  v       v         v        v        v
 hr_agent it_agent fin_agent leg_agent fallback
  |       |         |        |        |
  +-------+---------+--------+--------+
                    |
            evaluator_agent
                    |
                   END
```

| Nodo | Rol |
|---|---|
| `enterprise_router_node` | Clasifica el departamento con LLM |
| `hr_agent` | Políticas de RRHH, vacaciones, contratación |
| `it_agent` | Infraestructura, seguridad, SaaS corporativo |
| `finance_agent` | Presupuestos, gastos, reportes financieros |
| `legal_agent` | Contratos, compliance, NDA, regulaciones |
| `fallback_agent` | Consultas fuera de alcance |
| `evaluator_agent` | Audita la calidad de la respuesta (0–10 por dimensión) |

## Paso 1 — Elegí tu proveedor de LLM

En este primer paso definimos qué modelo va a razonar dentro del sistema. Todo el grafo usará el mismo objeto `llm`, así que cambiar `PROVIDER` cambia el motor de todos los nodos: router, agente especialista y evaluador.

Qué ocurre en esta celda:
- `PROVIDER` selecciona OpenAI, Gemini o Claude.
- Se instala solo la integración de LangChain necesaria para ese proveedor.
- La API key se pide con `getpass` para no dejar secretos escritos en el notebook.
- `temperature=0` reduce variabilidad y ayuda a que los checks sean más consistentes.
- El resultado esperado es una variable global `llm` lista para usar con `.invoke(...)`.

In [ ]:
PROVIDER = "openai"   # Cambiar a "gemini" o "claude" si queres probar otro proveedor.

# `os.environ` guarda las API keys como variables de entorno durante la sesion.
# `getpass` permite pegarlas sin que queden visibles en la salida del notebook.
import os
from getpass import getpass

# Cada rama deja lista una variable llamada `llm` con la misma interfaz:
# despues todos los nodos pueden llamar `invoke_llm(prompt)` sin saber que proveedor hay debajo.
if PROVIDER == "openai":
    # OpenAI es el camino recomendado para esta clase porque el costo y el modelo son faciles de ver en Langfuse.
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ").strip()
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    # Gemini usa otra integracion de LangChain, pero exponemos la misma interfaz de chat.
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ").strip()
    llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

elif PROVIDER == "claude":
    # Claude requiere `max_tokens` para fijar un limite razonable de salida.
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ").strip()
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0, max_tokens=1024)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

LLM listo → proveedor: openai


## Paso 2 — Instalar dependencias

Acá preparamos las piezas comunes del ejercicio, independientemente del proveedor de LLM.

Qué se usa y por qué:
- `langgraph`: permite modelar el sistema como un grafo de nodos conectados.
- `StateGraph`: define un flujo donde cada nodo recibe un estado y devuelve cambios parciales.
- `START` y `END`: marcan entrada y salida del grafo.
- `TypedDict`: documenta la forma esperada del estado compartido.
- `extract_json_object`: hace más robusto el parseo cuando el LLM devuelve JSON con texto extra o markdown.
- `score_0_10`: normaliza scores del evaluador para evitar valores fuera de rango.

In [ ]:
!pip install langgraph -q

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
import json, re, unicodedata

def extract_json_object(text: str) -> dict:
    """Extrae y parsea el primer objeto JSON que devuelva el LLM."""
    # Aunque pedimos JSON estricto, los LLMs a veces agregan ```json o texto extra.
    # Primero limpiamos fences de markdown y luego tomamos el bloque entre llaves.
    cleaned = re.sub(r"```[\w]*\n?", "", text).replace("```", "").strip()
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    payload = match.group(0) if match else cleaned
    return json.loads(payload)

def score_0_10(value, default: float = 5.0) -> float:
    """Convierte un score del LLM a float y lo limita al rango valido 0-10."""
    # El evaluador es otro LLM, por lo que nunca asumimos que devolvera un numero perfecto.
    # Si viene mal tipeado usamos `default`; si viene fuera de rango lo recortamos.
    try:
        number = float(value)
    except (TypeError, ValueError):
        number = default
    return max(0.0, min(10.0, number))

print("LangGraph listo.")

LangGraph listo.


## Paso 3 (opcional) — Langfuse para trazabilidad

> **Langfuse:** plataforma de observabilidad para sistemas LLM. Cada llamada al modelo puede quedar registrada con input, output, modelo, tokens, latencia y costo.

Este paso es opcional: el sistema multiagente funciona aunque Langfuse esté apagado. La idea didáctica es mostrar cómo un sistema que ya responde puede empezar a ser observable como en producción.

Qué se hace en esta celda:
- `LANGFUSE_ENABLED` controla si activamos trazabilidad o no.
- Se piden Public Key y Secret Key con `getpass`, sin hardcodearlas.
- `LANGFUSE_BASE_URL` se configura para la región correcta. Para US Cloud usamos `https://us.cloud.langfuse.com`.
- `auth_check()` valida credenciales antes de seguir.
- `CallbackHandler` conecta LangChain con Langfuse para que las llamadas reales al LLM aparezcan como `generation`.
- `invoke_llm(prompt)` centraliza las llamadas al modelo: si Langfuse está activo agrega callbacks; si no, ejecuta normal.

Para costos por token: además de activar esta celda, en Langfuse debe existir una definición de modelo cuyo `Match Pattern` coincida con el nombre enviado por LangChain, por ejemplo `(?i)^(gpt-4o-mini)$`.

In [ ]:
# Cambiar a True solo cuando quieras enviar trazas reales a Langfuse.
# En False, el ejercicio funciona igual y no consume observabilidad externa.
LANGFUSE_ENABLED = True   # cambiar a True si queres usar Langfuse

# La URL debe pertenecer a la misma region/proyecto que las API keys.
# US Cloud: https://us.cloud.langfuse.com | EU Cloud: https://cloud.langfuse.com
LANGFUSE_BASE_URL_DEFAULT = "https://us.cloud.langfuse.com"
langfuse_client = None
langfuse_handler = None

if LANGFUSE_ENABLED:
    try:
        # `langfuse` es el SDK; `langchain` habilita la integracion CallbackHandler.
        !pip install langfuse langchain -q
        from langfuse import Langfuse
        from langfuse.langchain import CallbackHandler
        # Pedimos secretos en runtime para no dejarlos escritos en el notebook.
        os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("Langfuse Public Key: ").strip()
        os.environ["LANGFUSE_SECRET_KEY"] = getpass("Langfuse Secret Key: ").strip()
        # Enter usa US Cloud por defecto; escribir otra URL solo si tu proyecto esta en otra region.
        langfuse_base_url = input(f"Langfuse Base URL [{LANGFUSE_BASE_URL_DEFAULT}]: ").strip() or LANGFUSE_BASE_URL_DEFAULT
        langfuse_base_url = langfuse_base_url.rstrip("/")
        os.environ["LANGFUSE_BASE_URL"] = langfuse_base_url
        os.environ["LANGFUSE_HOST"] = langfuse_base_url  # compatibilidad con SDKs/integraciones anteriores

        # Instanciamos el cliente despues de setear las variables de entorno.
        langfuse_client = Langfuse()
        if not langfuse_client.auth_check():
            raise RuntimeError("Langfuse rechazo las credenciales.")

        # Este handler registra cada llamada al LLM como generation: modelo, tokens, latencia y costo.
        langfuse_handler = CallbackHandler()
        print(f"Langfuse conectado correctamente en {os.environ['LANGFUSE_BASE_URL']}.")
    except Exception as e:
        print(f"Langfuse no disponible: {e}")
        print("Verifica que Public Key, Secret Key y Base URL pertenezcan al mismo proyecto y region.")
        print("El sistema funciona igual sin trazabilidad externa.")
        langfuse_client = None
        langfuse_handler = None
        LANGFUSE_ENABLED = False
else:
    print("Langfuse deshabilitado. Setear LANGFUSE_ENABLED=True para activar.")

def invoke_llm(prompt):
    # Punto unico de entrada al modelo: todos los nodos pasan por aca.
    # Si Langfuse esta activo, agregamos callbacks; si no, llamamos al LLM normal.
    config = {"callbacks": [langfuse_handler]} if LANGFUSE_ENABLED and langfuse_handler else None
    return llm.invoke(prompt, config=config) if config else llm.invoke(prompt)

Langfuse deshabilitado. Setear LANGFUSE_ENABLED=True para activar.


## Sección 1 — Knowledge bases corporativas

En esta sección armamos el conocimiento privado de cada departamento. Esto simula una base documental empresarial sin usar todavía embeddings ni vector stores.

Qué representa cada lista:
- `hr_kb`: políticas de recursos humanos.
- `it_kb`: accesos, VPN, seguridad y herramientas internas.
- `finance_kb`: gastos, reembolsos, presupuestos y reportes.
- `legal_kb`: NDAs, contratos, compliance y propiedad intelectual.

Por qué lo hacemos así: en un sistema RAG real estos textos vendrían de documentos, una base vectorial o una wiki corporativa. Acá los dejamos en listas para que el foco de la clase sea el flujo multiagente.

In [ ]:
hr_kb = [
    "Vacaciones: 15 días hábiles anuales para empleados con menos de 5 años; 20 días para más de 5 años.",
    "Licencia por maternidad/paternidad: 90 días para madre, 15 días para padre, remunerada al 100%.",
    "Proceso de contratación: 3 etapas (técnica, cultura fit, oferta). Duración estimada: 3-4 semanas.",
    "Beneficios: obra social premium, prepaga, ticket restaurante $5.000/mes, home office 2 días/semana.",
    "Revisión salarial: anual en diciembre, basada en performance review del trimestre anterior.",
    "Capacitación: presupuesto de $200.000/año por empleado para cursos, certificaciones o conferencias.",
    "Política de trabajo remoto: hasta 100% remoto previa aprobación del manager directo.",
]

it_kb = [
    "Acceso a sistemas: solicitar via ticket en el portal IT con aprobación del manager.",
    "VPN corporativa: usar GlobalProtect, reiniciar si hay problemas de conexión y validar MFA.",
    "Notebook corporativa: reposición en 3 años o ante falla mayor. Solicitar via formulario IT.",
    "SaaS aprobados: Google Workspace, Slack, Jira, GitHub, Figma, Zoom, Notion.",
    "Contraseñas: política de 12 caracteres mínimo, rotación cada 90 días, gestor LastPass disponible.",
    "Incidentes de seguridad: reportar inmediatamente a security@empresa.com y al manager.",
    "Backup: datos críticos en Google Drive corporativo con retención de 1 año.",
]

finance_kb = [
    "Gastos: reembolsables con ticket/factura y aprobación del manager hasta $50.000. Más: requiere VP.",
    "Presupuestos: proceso anual en noviembre. Solicitudes de ajuste mid-year via Finance.",
    "Viajes corporativos: vuelos en económica salvo trayectos >8hs. Hotel máximo $200/noche.",
    "Cierre mensual: los 5 primeros días hábiles del mes siguiente. Facturas deben entrar antes.",
    "Centro de costos: cada equipo tiene su propio CC. Validar con Finance antes de imputar.",
    "Tarjeta corporativa: disponible para directores y VPs. Gastos con justificación obligatoria.",
    "Reportes financieros: P&L mensual publicado el día 10 en el portal de Finance.",
]

legal_kb = [
    "NDAs: toda relación con terceros requiere NDA firmado. Plantillas en el portal Legal.",
    "Contratos con proveedores: revisión obligatoria por Legal para montos >$500.000.",
    "Compliance GDPR/LGPD: datos de usuarios EU/BR requieren consentimiento explícito y DPA.",
    "Propiedad intelectual: todo código desarrollado en horario laboral pertenece a la empresa.",
    "Política de conflictos de interés: declarar cualquier relación con competidores o proveedores.",
    "Litigios: cualquier notificación judicial debe escalarse inmediatamente al departamento Legal.",
    "Contratos de empleados: regidos por el Convenio Colectivo de Trabajo del sector tecnológico.",
]

enterprise_kbs = {
    "hr": hr_kb,
    "it": it_kb,
    "finance": finance_kb,
    "legal": legal_kb,
}

print("Knowledge bases corporativas cargadas:", list(enterprise_kbs.keys()))

Knowledge bases corporativas cargadas: ['hr', 'it', 'finance', 'legal']


## Sección 2 — State empresarial

El estado es el contrato compartido entre todos los nodos del grafo. Cada nodo recibe este diccionario, lee lo que necesita y devuelve solo los campos que quiere actualizar.

Campos principales:
- `query`: consulta original del empleado.
- `department`: decisión del router.
- `reason`: justificación breve del router.
- `agent_response`: respuesta del agente especialista o fallback.
- `eval_*`: métricas calculadas por el evaluador automático.

Este diseño permite seguir la historia completa de una consulta: entrada, decisión de routing, respuesta y evaluación.

In [ ]:
class EnterpriseState(TypedDict):
    query: str
    department: str        # "hr" | "it" | "finance" | "legal" | "unknown"
    reason: str
    agent_response: str    # respuesta del agente especialista
    eval_score: float      # promedio de las 4 dimensiones (0.0–10.0)
    eval_relevance: float
    eval_completeness: float
    eval_accuracy: float
    eval_clarity: float
    eval_feedback: str     # feedback textual del evaluador

## Sección 3 — Nodos del sistema

A partir de acá definimos el comportamiento de cada parte del grafo.

Orden lógico:
1. `enterprise_router_node` decide qué departamento debe responder usando LLM y un respaldo determinístico por keywords.
2. Un agente especialista usa solo su knowledge base para contestar.
3. `fallback_agent` responde cuando la consulta no corresponde a ningún departamento.
4. `evaluator_agent` audita la respuesta final.

> **Evaluador automático:** usa el mismo LLM para auditar la respuesta de otro agente. El prompt pide un JSON estructurado con scores y feedback. Este patrón se llama `LLM-as-a-judge`.

Punto importante: todos los nodos que usan modelo llaman a través de `invoke_llm(...)`, no directamente con `llm.invoke(...)`. Eso permite activar Langfuse sin reescribir cada agente. El respaldo por keywords no llama al modelo; solo estabiliza casos claros para que la demo no falle por una clasificación aislada.

In [ ]:
DEPARTMENT_KEYWORDS = {
    "hr": ["vacaciones", "capacitacion", "capacitaci", "beneficio", "contratacion", "salario", "remoto"],
    "it": ["vpn", "saas", "herramienta", "acceso", "sistema", "password", "contrasena", "seguridad", "notebook"],
    "finance": ["reembolso", "gasto", "factura", "presupuesto", "viaje", "tarjeta", "financiero", "p&l"],
    "legal": ["nda", "contrato", "proveedor", "compliance", "gdpr", "lgpd", "legal", "litigio"],
}

def normalize_text(text: str) -> str:
    # Lowercase + sin acentos: asi `capacitación` y `capacitacion` matchean igual.
    lowered = text.lower()
    return "".join(ch for ch in unicodedata.normalize("NFD", lowered) if unicodedata.category(ch) != "Mn")

def keyword_department(query: str) -> tuple[str, str]:
    # Respaldo deterministico para consultas obvias. No reemplaza al LLM; evita que un caso claro rompa la demo.
    normalized = normalize_text(query)
    for department, keywords in DEPARTMENT_KEYWORDS.items():
        for keyword in keywords:
            if keyword in normalized:
                return department, f"Keyword routing: '{keyword}' indica {department}."
    return "unknown", "No hubo coincidencia deterministica."

def enterprise_router_node(state: EnterpriseState) -> dict:
    # El router solo decide el departamento; no responde la consulta final.
    # Devuelve `department` y `reason`, que luego usa LangGraph para elegir el siguiente nodo.
    prompt = (
        "Sos el router de un sistema de soporte empresarial corporativo.\n"
        "Clasificá la consulta en exactamente un departamento:\n"
        "- 'hr': recursos humanos, vacaciones, beneficios, contratación, desvinculación\n"
        "- 'it': tecnología, infraestructura, acceso a sistemas, seguridad informática\n"
        "- 'finance': presupuestos, gastos, reembolsos, facturas, reportes financieros\n"
        "- 'legal': contratos, compliance, NDA, propiedad intelectual, litigios\n"
        "- 'unknown': consultas que no corresponden a ningún departamento\n\n"
        "Respondé con JSON: {\"department\": \"...\", \"reason\": \"...\"}\n"
        "Solo department y reason, sin markdown.\n\n"
        f"Consulta: {state['query']}"
    )
    # Usamos el helper para que esta llamada tambien quede trazada si Langfuse esta activo.
    response = invoke_llm(prompt)
    try:
        # Parseamos JSON de forma robusta porque el LLM puede envolverlo en markdown.
        data = extract_json_object(response.content)
        department = data.get("department", "unknown").lower()
        reason = data.get("reason", "")
    except Exception:
        department = "unknown"
        reason = response.content.strip()
    # Cualquier etiqueta fuera del set permitido va a fallback para evitar rutas inexistentes.
    if department not in ("hr", "it", "finance", "legal"):
        department = "unknown"

    # Si el LLM no pudo clasificar, aplicamos el respaldo de keywords para casos evidentes.
    if department == "unknown":
        keyword_dept, keyword_reason = keyword_department(state["query"])
        if keyword_dept != "unknown":
            department = keyword_dept
            reason = keyword_reason
    return {"department": department, "reason": reason}


print("Router definido.")

Router definido.


In [ ]:
def _enterprise_rag_agent(dept: str, display: str, state: EnterpriseState) -> dict:
    # Funcion reutilizable para no duplicar el mismo prompt en HR, IT, Finance y Legal.
    # `dept` elige la knowledge base; `display` solo mejora el texto visible del prompt.
    context = "\n".join(enterprise_kbs[dept])
    response = invoke_llm(
        f"Sos el agente de {display} de una empresa tecnológica.\n"
        "Respondé la consulta del empleado usando únicamente el contexto provisto.\n"
        "Si la información no está en el contexto, indicalo claramente sin inventar.\n\n"
        f"Política y procedimientos de {display}:\n{context}\n\n"
        f"Consulta del empleado: {state['query']}\n\n"
        "Respondé de forma clara, precisa y profesional en español."
    )
    # El nodo no pisa todo el estado: devuelve solo el campo que acaba de producir.
    return {"agent_response": response.content.strip()}


def hr_agent(state: EnterpriseState) -> dict:
    return _enterprise_rag_agent("hr", "Recursos Humanos", state)


def it_agent(state: EnterpriseState) -> dict:
    return _enterprise_rag_agent("it", "Tecnología", state)


def finance_agent(state: EnterpriseState) -> dict:
    return _enterprise_rag_agent("finance", "Finanzas", state)


def legal_agent(state: EnterpriseState) -> dict:
    return _enterprise_rag_agent("legal", "Legal", state)


def fallback_agent(state: EnterpriseState) -> dict:
    # Fallback no llama al LLM: responde de forma deterministica cuando el router no encontro area.
    return {
        "agent_response": (
            "Tu consulta no pudo ser asignada a un departamento específico. "
            "Por favor contactá directamente a: hr@empresa.com (RRHH), it@empresa.com (IT), "
            "finance@empresa.com (Finanzas) o legal@empresa.com (Legal)."
        )
    }


def enterprise_router(state: EnterpriseState) -> str:
    # Esta funcion no es un nodo: es la condicion que decide hacia que nodo ir.
    return {
        "hr":      "hr_agent",
        "it":      "it_agent",
        "finance": "finance_agent",
        "legal":   "legal_agent",
    }.get(state["department"], "fallback_agent")


print("Agentes especialistas definidos.")

Agentes especialistas definidos.


In [ ]:
def evaluator_agent(state: EnterpriseState) -> dict:
    """
    LLM-as-a-judge: evalúa la respuesta del agente en 4 dimensiones (0-10 cada una).
    Devuelve scores numéricos y feedback textual.
    """
    # El evaluador recibe la consulta original y la respuesta final.
    # No decide el departamento ni reescribe la respuesta; solo calcula calidad.
    eval_prompt = (
        "Sos un evaluador de calidad de respuestas de sistemas multiagente corporativos.\n"
        "Evaluá la siguiente respuesta en 4 dimensiones del 0 al 10:\n\n"
        "- relevance: ¿la respuesta es relevante a la consulta del empleado?\n"
        "- completeness: ¿la respuesta cubre todos los aspectos de la consulta?\n"
        "- accuracy: ¿la información es precisa y no contiene errores?\n"
        "- clarity: ¿la respuesta es clara y fácil de entender?\n\n"
        "Respondé SOLO con JSON válido sin markdown:\n"
        "{\"relevance\": N, \"completeness\": N, \"accuracy\": N, \"clarity\": N, \"feedback\": \"...\"}"
        f"\n\nConsulta original: {state['query']}\n"
        f"Respuesta del agente: {state['agent_response']}"
    )

    try:
        # Esta es otra llamada real al modelo, por lo tanto tambien se traza en Langfuse.
        response = invoke_llm(eval_prompt)
        # Extraemos JSON y protegemos el pipeline contra respuestas con formato imperfecto.
        data = extract_json_object(response.content)
        relevance     = score_0_10(data.get("relevance", 5))
        completeness  = score_0_10(data.get("completeness", 5))
        accuracy      = score_0_10(data.get("accuracy", 5))
        clarity       = score_0_10(data.get("clarity", 5))
        feedback      = str(data.get("feedback", ""))
        score         = round((relevance + completeness + accuracy + clarity) / 4, 2)
    except Exception as e:
        # Si el evaluador falla, no rompemos toda la demo: devolvemos un score neutro y feedback del error.
        relevance = completeness = accuracy = clarity = 5.0
        score = 5.0
        feedback = f"Error al evaluar: {e}"

    # Devolvemos solo campos de evaluacion para que LangGraph los fusione con el estado anterior.
    return {
        "eval_score":        score,
        "eval_relevance":    relevance,
        "eval_completeness": completeness,
        "eval_accuracy":     accuracy,
        "eval_clarity":      clarity,
        "eval_feedback":     feedback,
    }


print("Evaluador definido.")

Evaluador definido.


## Sección 4 — Compilar el grafo

En esta sección conectamos los nodos definidos antes y convertimos funciones sueltas en un flujo ejecutable.

Qué hace el grafo:
- Empieza en `START`.
- Siempre pasa primero por `enterprise_router_node`.
- Usa `add_conditional_edges` para elegir el agente correcto según `department`.
- Todos los agentes, incluido fallback, terminan en `evaluator_agent`.
- El evaluador termina el proceso en `END`.

El resultado de `graph.compile()` es `app`, el objeto que vamos a invocar con consultas reales.

In [ ]:
# Creamos un grafo cuyo estado compartido tiene la forma EnterpriseState.
graph = StateGraph(EnterpriseState)

# Registramos cada funcion Python como un nodo con nombre estable.
graph.add_node("enterprise_router_node", enterprise_router_node)
graph.add_node("hr_agent",               hr_agent)
graph.add_node("it_agent",               it_agent)
graph.add_node("finance_agent",          finance_agent)
graph.add_node("legal_agent",            legal_agent)
graph.add_node("fallback_agent",         fallback_agent)
graph.add_node("evaluator_agent",        evaluator_agent)

# El primer paso siempre es clasificar la consulta.
graph.add_edge(START, "enterprise_router_node")

# Despues del router, LangGraph llama `enterprise_router` para decidir la rama.
graph.add_conditional_edges(
    "enterprise_router_node",
    enterprise_router,
    {
        "hr_agent":      "hr_agent",
        "it_agent":      "it_agent",
        "finance_agent": "finance_agent",
        "legal_agent":   "legal_agent",
        "fallback_agent": "fallback_agent",
    },
)
# Todas las ramas convergen en el mismo evaluador para tener metricas comparables.
for node in ["hr_agent", "it_agent", "finance_agent", "legal_agent", "fallback_agent"]:
    graph.add_edge(node, "evaluator_agent")

graph.add_edge("evaluator_agent", END)

# `compile()` valida la estructura y devuelve un objeto ejecutable con `.invoke(...)`.
app = graph.compile()
print("Grafo compilado.")

Grafo compilado.


## Demo — Consultas de empleados con evaluación automática

Esta celda ejecuta el sistema como lo usaría un empleado. Cada consulta sigue el mismo camino: router, agente especialista o fallback, evaluador y salida final.

Qué observar en la salida:
- `Departamento`: confirma si el router eligió el área correcta.
- `Respuesta`: muestra qué contestó el agente con su knowledge base.
- `Score total`: resume la evaluación automática.
- `rel`, `comp`, `acc`, `clar`: muestran relevancia, completitud, precisión y claridad.
- `Feedback`: explica qué vio el evaluador.

Si Langfuse está activo, al final se llama `flush()` para forzar el envío de eventos antes de que termine la celda.

In [ ]:
# Estado base: LangGraph recibira una copia con la query real en cada iteracion.
# Dejamos todos los campos inicializados para que el contrato sea visible para estudiantes.
EMPTY = {
    "query": "", "department": "", "reason": "", "agent_response": "",
    "eval_score": 0.0, "eval_relevance": 0.0, "eval_completeness": 0.0,
    "eval_accuracy": 0.0, "eval_clarity": 0.0, "eval_feedback": "",
}

# Consultas de prueba: una por cada departamento y una fuera de alcance para probar fallback.
queries = [
    "¿Cuántos días de vacaciones tengo si llevo 6 años en la empresa?",
    "No puedo conectarme a la VPN desde mi casa",
    "Necesito reembolso de un gasto de $80.000 de un viaje de trabajo",
    "Un proveedor me pide firmar un contrato sin revisión legal, ¿puedo hacerlo?",
    "¿Dónde puedo conseguir café gratis en la oficina?",
]

for q in queries:
    # Cada query arranca con estado limpio para no mezclar resultados entre ejecuciones.
    state_input = {**EMPTY, "query": q}
    r = app.invoke(state_input)

    print(f"\nConsulta:    {q}")
    print(f"Departamento: {r['department']}")
    print(f"Respuesta:   {r['agent_response'][:100]}...")
    print(f"Score total: {r['eval_score']:.1f}/10  "
          f"(rel={r['eval_relevance']:.0f} comp={r['eval_completeness']:.0f} "
          f"acc={r['eval_accuracy']:.0f} clar={r['eval_clarity']:.0f})")
    print(f"Feedback:    {r['eval_feedback'][:80]}")
    print("-" * 70)

# En notebooks conviene forzar flush: si la celda termina rapido, evitamos perder eventos pendientes.
if LANGFUSE_ENABLED and langfuse_client:
    langfuse_client.flush()


Consulta:    ¿Cuántos días de vacaciones tengo si llevo 6 años en la empresa?
Departamento: hr
Respuesta:   Estimado/a [Nombre del Empleado],

Si llevas 6 años en la empresa, te corresponden 20 días hábiles d...
Score total: 9.8/10  (rel=10 comp=10 acc=10 clar=9)
Feedback:    La respuesta es muy relevante y completa, proporcionando la información exacta s
----------------------------------------------------------------------

Consulta:    No puedo conectarme a la VPN desde mi casa
Departamento: it
Respuesta:   Para resolver el problema de conexión a la VPN desde tu casa, te recomiendo seguir estos pasos:

1. ...
Score total: 9.2/10  (rel=10 comp=8 acc=10 clar=9)
Feedback:    La respuesta es muy relevante y precisa, proporcionando pasos claros para resolv
----------------------------------------------------------------------

Consulta:    Necesito reembolso de un gasto de $80.000 de un viaje de trabajo
Departamento: finance
Respuesta:   Estimado/a [Nombre del empleado],

Para procesar 

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

Estos checks son pruebas de humo: no reemplazan una suite formal, pero sirven para validar que el flujo principal sigue funcionando.

Qué verifican:
- Que consultas claras lleguen al departamento correcto.
- Que el evaluador siempre corra y devuelva un score.
- Que incluso una consulta fuera de alcance pase por fallback y evaluación.

Importante para clase: estos checks consumen tokens porque invocan el LLM varias veces. Si un modelo responde de forma inesperada, el assert puede fallar aunque el grafo esté bien armado; en ese caso revisamos el prompt del router o la consulta de prueba.

In [ ]:
def run_checks():
    # Estos checks ejercitan el flujo completo, no funciones aisladas.
    # Cada assert obliga a pasar por router, agente y evaluador.
    empty = {
        "query": "", "department": "", "reason": "", "agent_response": "",
        "eval_score": 0.0, "eval_relevance": 0.0, "eval_completeness": 0.0,
        "eval_accuracy": 0.0, "eval_clarity": 0.0, "eval_feedback": "",
    }

    # RRHH: la palabra capacitacion deberia activar la base de politicas de empleados.
    r1 = app.invoke({**empty, "query": "¿cuántos días de capacitación pagada tengo?"})
    assert r1["department"] == "hr", f"esperaba hr: {r1['department']}"
    assert isinstance(r1["eval_score"], float) and r1["eval_score"] >= 0
    assert isinstance(r1["eval_feedback"], str) and len(r1["eval_feedback"]) > 0

    # IT: SaaS y herramientas aprobadas pertenecen a tecnologia corporativa.
    r2 = app.invoke({**empty, "query": "¿qué herramientas SaaS están aprobadas para usar?"})
    assert r2["department"] == "it", f"esperaba it: {r2['department']}"

    # Finance: reembolso y viaje corporativo pertenecen a gastos y finanzas.
    r3 = app.invoke({**empty, "query": "¿cómo proceso el reembolso de un viaje corporativo?"})
    assert r3["department"] == "finance", f"esperaba finance: {r3['department']}"

    # Legal: NDA y proveedor deben ir a revision legal.
    r4 = app.invoke({**empty, "query": "un proveedor quiere que firme un NDA esta semana"})
    assert r4["department"] == "legal", f"esperaba legal: {r4['department']}"

    # verificar que el evaluador siempre corre (incluso en fallback)
    r5 = app.invoke({**empty, "query": "¿cuál es el mejor restaurante cerca?"})
    assert r5["eval_score"] >= 0, "evaluador no corrió"

    print("Checks E19 OK")

run_checks()

## ¿Qué viste en este caso?

- Un sistema empresarial completo puede modelarse como un grafo con múltiples nodos especializados conectados secuencialmente.
- El patrón **LLM-as-a-judge** permite auditar automáticamente la calidad de las respuestas sin intervención humana.
- **Langfuse** agrega observabilidad de producción: cada consulta queda registrada con inputs, outputs y métricas.

| Concepto | Implementación en E19 |
|---|---|
| 4 departamentos + fallback | 5 agentes RAG en nodos separados |
| Evaluador automático | `evaluator_agent` → LLM devuelve JSON con 4 scores |
| LLM-as-a-judge | El mismo LLM que responde también evalúa (roles distintos) |
| Langfuse (opcional) | Callback de LangChain para registrar generaciones, tokens y costos |
| Flujo post-agente | Todos los agentes → `evaluator_agent` → `END` |

## Próximo ejercicio

En **E20** vas a ver un sistema de programación con 5 tecnologías especializadas (React, Angular, NestJS, Express, Python) y un router que detecta área y tecnología en una sola clasificación.